# 03 - Análisis Principal

Este notebook desarrolla el análisis principal del proyecto GESAssist.

Pregunta de análisis:

**¿Qué patrones se observan en los diagnósticos clasificados como GES y No GES considerando el diagnóstico y la edad del paciente?**


In [1]:
import pandas as pd
import plotly.express as px

df = pd.read_csv("../data/dataset_limpio.csv")
df.head()

,id,diagnostic,age,ges
0,1,RAQUIESTENOSIS L4-L5 Y DISCOPATIA SEVERA L5-S1,67,False
1,10,ABSCESO OVARICO,50,False
2,100,CALCULO DEL URETER,57,False
3,101,CALCULO DEL URETER,28,False
4,102,CALCULO DEL URETER,54,False


## Métricas generales

In [2]:
total_casos = len(df)
casos_ges = int(df["ges"].sum())
casos_no_ges = total_casos - casos_ges
porcentaje_ges = casos_ges / total_casos * 100

print("Total de casos:", total_casos)
print("Casos GES:", casos_ges)
print("Casos No GES:", casos_no_ges)
print("Porcentaje GES:", round(porcentaje_ges, 2), "%")

Total de casos: 941
Casos GES: 260
Casos No GES: 681
Porcentaje GES: 27.63 %


## Proporción de casos GES y No GES

In [3]:
df_clasificacion = df["ges"].map({True: "GES", False: "No GES"}).value_counts().reset_index()
df_clasificacion.columns = ["clasificacion", "cantidad"]

fig = px.pie(
    df_clasificacion,
    names="clasificacion",
    values="cantidad",
    title="Proporción de casos GES y No GES"
)
fig.show()

## Diagnósticos GES más frecuentes

In [4]:
diagnosticos_ges = (
    df[df["ges"] == True]["diagnostic"]
    .value_counts()
    .head(15)
    .reset_index()
)

diagnosticos_ges.columns = ["diagnostic", "cantidad"]
diagnosticos_ges

,diagnostic,cantidad
0,CATARATA OD,20
1,CATARATA OI,20
2,CATARATA ODI SAN JOSE,18
3,CATARATA OI SAN JOSE,12
4,COLELITIASIS,11
5,HIPERPLASIA DE LA PROSTATA HPB,11
6,IRC,7
7,CA DE MAMA IZQ,6
8,DR OI,5
9,CATARATA OD OI,5


In [5]:
fig = px.bar(
    diagnosticos_ges,
    x="cantidad",
    y="diagnostic",
    orientation="h",
    text="cantidad",
    title="Top 15 diagnósticos GES más frecuentes"
)
fig.update_layout(yaxis={"categoryorder": "total ascending"})
fig.show()

## Diagnósticos No GES más frecuentes

In [6]:
diagnosticos_no_ges = (
    df[df["ges"] == False]["diagnostic"]
    .value_counts()
    .head(15)
    .reset_index()
)

diagnosticos_no_ges.columns = ["diagnostic", "cantidad"]
diagnosticos_no_ges

,diagnostic,cantidad
0,COLELITIASIS,27
1,CBC,17
2,CX DE TERCEROS MOLARES,16
3,FIMOSIS,14
4,CALCULO DEL URETER,11
5,MULTIPARIDAD,11
6,QUISTE,11
7,HERNIA INGUINAL DERECHA,10
8,VARICES PIERNA DERECHA,9
9,CALCULO DEL RINON,8


In [7]:
fig = px.bar(
    diagnosticos_no_ges,
    x="cantidad",
    y="diagnostic",
    orientation="h",
    text="cantidad",
    title="Top 15 diagnósticos No GES más frecuentes"
)
fig.update_layout(yaxis={"categoryorder": "total ascending"})
fig.show()

## Análisis por grupo etario

In [8]:
df["grupo_edad"] = pd.cut(
    df["age"],
    bins=[0, 18, 30, 45, 60, 75, 100],
    labels=["0-18", "19-30", "31-45", "46-60", "61-75", "76+"]
)

grupo_edad = (
    df.groupby(["grupo_edad", "ges"], observed=False)
    .size()
    .reset_index(name="cantidad")
)

grupo_edad["clasificacion"] = grupo_edad["ges"].map({True: "GES", False: "No GES"})
grupo_edad

,grupo_edad,ges,cantidad,clasificacion
0,0-18,False,109,No GES
1,0-18,True,2,GES
2,19-30,False,98,No GES
3,19-30,True,5,GES
4,31-45,False,89,No GES
5,31-45,True,27,GES
6,46-60,False,167,No GES
7,46-60,True,76,GES
8,61-75,False,157,No GES
9,61-75,True,96,GES


In [9]:
fig = px.bar(
    grupo_edad,
    x="grupo_edad",
    y="cantidad",
    color="clasificacion",
    barmode="group",
    title="Casos GES y No GES por grupo etario",
    labels={"grupo_edad": "Grupo de edad", "cantidad": "Cantidad"}
)
fig.show()

## Hallazgo principal

El dataset presenta una mayor cantidad de casos No GES que GES. Sin embargo, dentro de los casos GES se observan diagnósticos recurrentes como catarata, cáncer de mama, cáncer de colon, insuficiencia renal crónica y otros diagnósticos específicos.

Un hallazgo relevante es que la clasificación GES no depende únicamente del texto del diagnóstico. También puede estar relacionada con la edad del paciente y con criterios específicos de cobertura.

Este análisis permite construir un dashboard que resume la información y facilita la exploración por diagnóstico, edad y clasificación.
